# 01. Analise exploratoria e qualidade dos dados

Objetivo: medir, em cada um dos 5 arquivos, os tipos de problema, o volume afetado e o impacto potencial.
Os arquivos sao lidos como texto puro (sem converter nada) para nao esconder problemas na leitura.
A logica de limpeza fica em `src/cristalux/cleaning/`; este notebook so mostra e comenta.

In [1]:
import pandas as pd
from pathlib import Path
from cristalux.config import settings
from cristalux.cleaning import parsers as p
from cristalux.cleaning.base import ler_csv
from cristalux.pipeline.run import ARQUIVOS, limpar_tudo, relatorio_qualidade

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200, "display.max_colwidth", 60)
raws, res = limpar_tudo(settings.data_dir)

## Visao geral

In [2]:
visao = pd.DataFrame({
    n: {"linhas": len(raws[n]), "colunas": raws[n].shape[1] - 1,
        "linhas_totalmente_vazias": int((raws[n].drop(columns="linha_origem").apply(lambda c: c.map(p.limpar)).isna().sum(axis=1) >= raws[n].shape[1] - 2).sum()),
        "linhas_duplicadas_exatas": int(raws[n].drop(columns="linha_origem").duplicated().sum()),
        "ids_repetidos": int(raws[n].iloc[:, 1].duplicated().sum())}
    for n in ARQUIVOS}).T
visao

,linhas,colunas,linhas_totalmente_vazias,linhas_duplicadas_exatas,ids_repetidos
compradores,30,12,1,0,0
vendedores,20,10,0,1,1
estoque_logistica,30,11,2,0,0
decisoes,23,8,2,1,1
vendas,190,13,3,12,12


Observacoes:

- `vendas` e o maior arquivo (190 linhas) e o que mais concentra problemas.
- Linhas duplicadas exatas aparecem em `vendas`, `vendedores` e `decisoes`.
- Registros "vazios" (sem informacao util) aparecem em todos os arquivos, inclusive 3 em `vendas`, com variantes: celulas em branco, `N/A` e espaco.

## Valores ausentes e marcadores de nulo

Contagem de celulas vazias, `N/A`, `null` e `DESCONHECIDO`, por coluna.

In [3]:
for n in ARQUIVOS:
    df = raws[n].drop(columns="linha_origem")
    nulos = df.apply(lambda c: c.map(p.limpar).isna().sum())
    print(f"--- {n}")
    print(nulos[nulos > 0].to_string() if (nulos > 0).any() else "sem nulos")

--- compradores
razao_social      2
cnpj              1
cidade            2
estado            2
segmento          2
porte             2
contato           4
email             4
data_cadastro     3
limite_credito    2
status            2
--- vendedores
nome             1
email            1
telefone         1
regiao           1
data_admissao    1
meta_mensal      1
comissao_pct     1
status           1
supervisor       1
--- estoque_logistica
nome_produto            2
categoria               2
estoque_atual           2
estoque_minimo          2
ultima_reposicao        4
lead_time_dias          3
fornecedor              5
custo_unitario          2
localizacao_deposito    5
status                  2
--- decisoes
data           2
tipo           2
descricao      2
responsavel    2
impacto        2
resultado      2
observacoes    2
--- vendas
id_venda            1
data               28
id_vendedor         3
id_comprador        3
produto             3
categoria           3
quantidade          3

## Formatos de data

O mesmo campo aparece em varios formatos, alguns invalidos.

In [4]:
from cristalux.cleaning.quality import formatos_de_data
colunas_data = {"vendas": "data", "compradores": "data_cadastro", "vendedores": "data_admissao",
                "estoque_logistica": "ultima_reposicao", "decisoes": "data"}
pd.DataFrame({n: formatos_de_data(raws[n][c]) for n, c in colunas_data.items()}).fillna(0).astype(int)

,vendas,compradores,vendedores,estoque_logistica,decisoes
dd-mm-aa,4,0,0,0,0
aaaa/mm/dd,12,3,4,4,0
dd-mm-aaaa,13,4,0,3,0
dd/mm/aaaa,90,14,11,13,21
dd.mm.aa,6,0,0,0,0
vazio,28,3,1,4,2
aaaa-mm-dd,28,5,4,3,0
dd.mm.aaaa,4,1,0,3,0
mmm/aaaa,5,0,0,0,0


In [5]:
v = raws["vendas"]
invalidas = v[v["data"].map(lambda t: p.parse_data(t)[1] not in (None,))]
print(len(invalidas), "datas de venda que nao viram data valida")
invalidas["data"].map(lambda t: t if p.limpar(t) else "(vazia)").value_counts()

53 datas de venda que nao viram data valida


data
(vazia)       28
31/11/2023     6
11/22/2022     6
Jul/2021       5
30/02/2021     5
00/12/2022     3
Name: count, dtype: int64

Datas validas que ainda assim merecem atencao: formatos com hifen (`03-01-2020`) podem ser dia-mes ou mes-dia.
No arquivo, essas mesmas datas aparecem escritas em outros formatos como 1 de marco de 2020, o que sugere mes-dia,
mas a evidencia nao e conclusiva. A regra adotada e dia-mes (convencao brasileira) com o flag `data_ambigua`.

## Grafias diferentes para o mesmo valor

In [6]:
def variantes(serie, funcao):
    brutas = serie.map(p.limpar).dropna()
    return {"grafias_brutas": brutas.nunique(), "valores_reais": brutas.map(funcao).dropna().nunique()}

pd.DataFrame({
    "vendas.regiao": variantes(raws["vendas"]["regiao"], p.parse_uf),
    "vendas.status": variantes(raws["vendas"]["status"], p.parse_status_venda),
    "vendas.produto": variantes(raws["vendas"]["produto"], p.parse_produto),
    "vendas.categoria": variantes(raws["vendas"]["categoria"], p.parse_categoria),
    "compradores.estado": variantes(raws["compradores"]["estado"], p.parse_uf),
    "vendedores.regiao": variantes(raws["vendedores"]["regiao"], p.parse_uf),
}).T

,grafias_brutas,valores_reais
vendas.regiao,35,8
vendas.status,14,4
vendas.produto,27,10
vendas.categoria,11,4
compradores.estado,19,9
vendedores.regiao,18,8


## Numeros escritos como texto

In [7]:
v = raws["vendas"]
pd.Series({
    "valor_unitario com 'R$'": int(v["valor_unitario"].str.contains("R\\$").sum()),
    "valor_total com 'R$'": int(v["valor_total"].str.contains("R\\$").sum()),
    "desconto como 'Sim'": int((v["desconto"].str.strip().str.lower() == "sim").sum()),
    "desconto com '%'": int(v["desconto"].str.contains("%").sum()),
    "quantidade negativa": int(v["quantidade"].map(p.parse_inteiro).lt(0).sum()),
    "valor_total divergente de qtd x preco (sem desconto)": int(
        ((v["quantidade"].map(p.parse_numero) * v["valor_unitario"].map(p.parse_numero)
          - v["valor_total"].map(p.parse_numero)).abs() > 0.02).sum()),
})

valor_unitario com 'R$'                                 49
valor_total com 'R$'                                    49
desconto como 'Sim'                                     17
desconto com '%'                                        42
quantidade negativa                                     21
valor_total divergente de qtd x preco (sem desconto)    31
dtype: int64

O ultimo item inclui vendas com desconto legitimo. A comparacao final, com desconto, e feita na limpeza (flag `valor_total_divergente`).

## Relatorio de qualidade

Gerado por codigo a partir das limpezas; e a mesma fonte que a tool `relatorio_qualidade` do agente usa.

In [8]:
from cristalux.cleaning.quality import salvar
rel = relatorio_qualidade(raws, res)
salvar(rel, settings.reports_dir)
tabela = pd.DataFrame([{"arquivo": a["arquivo"], **pr} for a in rel["arquivos"] for pr in a["problemas"]])
print(rel["resumo"])
tabela[["arquivo", "tipo", "volume", "percentual", "impacto"]]

{'total_linhas_brutas': 293, 'total_linhas_limpas': 164, 'total_quarentena': 129}


,arquivo,tipo,volume,percentual,impacto
0,compradores,duplicata_cnpj,13,43.3,Mesmo cliente com dois cadastros; fragmenta o historico ...
1,compradores,registro_vazio,2,6.7,Linha sem informacao util; inflaria contagens de registr...
2,compradores,cnpj_invalido,15,100.0,CNPJ com digitos verificadores invalidos; cadastro pode ...
3,compradores,data_cadastro_invalida,1,6.7,Data de cadastro invalida.
4,compradores,email_ausente,1,6.7,Sem e-mail de contato.
5,compradores,grafias_diferentes:estado,19,NaN,O mesmo valor aparece escrito de varias formas; agrupame...
6,compradores,grafias_diferentes:segmento,8,NaN,O mesmo valor aparece escrito de varias formas; agrupame...
7,compradores,grafias_diferentes:porte,9,NaN,O mesmo valor aparece escrito de varias formas; agrupame...
8,vendedores,registro_vazio,1,5.0,Linha sem informacao util; inflaria contagens de registr...
9,vendedores,duplicata_exata,1,5.0,Linha repetida; duplicaria faturamento e contagem de ven...


## Conclusao

Nenhum dos 5 arquivos esta pronto para analise. O problema mais grave para o objetivo do teste (faturamento mensal por vendedor)
esta em `vendas`: cerca de metade das linhas tem pelo menos um defeito que impede o uso (data ausente ou impossivel,
quantidade negativa, duplicata, comprador inexistente). Por isso a limpeza nao corrige por inferencia: ela separa o que
pode ser usado do que fica em quarentena, com o motivo.